# Shot profiles: where shots are worth the most, and who beats the league there

Every shot the league tracks is tagged with a court zone. For each league this notebook takes the
latest season with a meaningful number of shots (early in a season it falls back to the last full
one), measures what a shot in each zone is worth in points per 100 attempts, and compares the
highest-volume shooters with the league in every zone. Every number comes from `fct_shot`, the
same table behind the shot charts on the site's player pages.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
from fs import connect, describe, leagues, save_findings

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
con = connect()
LEAGUES = leagues(con)
LABEL = {'wnba': 'WNBA', 'nba': 'NBA'}
COLOR = {'wnba': '#b23a0a', 'nba': '#1a7f4b'}
findings = {}
print(describe(con), 'Leagues:', ', '.join(LABEL[lg] for lg in LEAGUES))

In [ ]:
def shot_season(lg):
    seasons = con.sql(f"select season, count(*) n from fct_shot where league = '{lg}' group by 1 order by 1 desc").fetchall()
    if len(seasons) > 1 and seasons[0][1] < 0.25 * seasons[1][1]:
        return seasons[1][0], f'{seasons[0][0]} is only {seasons[0][1]:,} shots in, so this uses {seasons[1][0]}'
    return seasons[0][0], None

zones = {}
for lg in LEAGUES:
    season, note = shot_season(lg)
    zones[lg] = (season, note, con.sql(f'''
        select shot_zone_basic as zone, count(*) as attempts,
               round(100.0 * avg(case when shot_made then 1 else 0 end), 1) as fg_pct,
               round(100.0 * avg(case when shot_made then (case when is_three then 3 else 2 end) else 0 end), 1) as pp100,
               round(100.0 * count(*) / sum(count(*)) over (), 1) as share_pct
        from fct_shot
        where league = '{lg}' and season = '{season}' and shot_zone_basic <> 'Backcourt'
        group by 1 order by pp100 desc
    ''').df())
    print(f"{LABEL[lg]} {season}" + (f" ({note})" if note else ''))
    display(zones[lg][2])

In [ ]:
fig, axes = plt.subplots(1, len(LEAGUES), figsize=(6 * len(LEAGUES), 4), squeeze=False)
for ax, lg in zip(axes[0], LEAGUES, strict=False):
    season, _, z = zones[lg]
    ax.barh(z['zone'], z['pp100'], color=COLOR[lg])
    ax.invert_yaxis()
    ax.set_xlabel('Points per 100 attempts')
    ax.set_title(f'{LABEL[lg]} {season}')
plt.tight_layout()
plt.show()

## The highest-volume shooters, zone by zone

For the ten players with the most attempts in each league, each zone's points per 100 attempts
against the league's in that zone (zones with fewer than 15 attempts left blank), and the
weighted overall edge.

In [ ]:
edges = {}
for lg in LEAGUES:
    season, _, _ = zones[lg]
    players = con.sql(f'''
        with per_zone as (
            select player_id, shot_zone_basic as zone, count(*) as attempts,
                   sum(case when shot_made then (case when is_three then 3 else 2 end) else 0 end) as points
            from fct_shot
            where league = '{lg}' and season = '{season}' and shot_zone_basic <> 'Backcourt'
            group by 1, 2 having count(*) >= 15
        ),
        lg_zone as (
            select shot_zone_basic as zone,
                   100.0 * avg(case when shot_made then (case when is_three then 3 else 2 end) else 0 end) as league_pp100
            from fct_shot where league = '{lg}' and season = '{season}' and shot_zone_basic <> 'Backcourt'
            group by 1
        ),
        top as (select player_id, sum(attempts) as attempts from per_zone group by 1 order by 2 desc limit 10)
        select coalesce(p.full_name, z.player_id) as player, z.zone, z.attempts,
               round(100.0 * z.points / z.attempts - l.league_pp100, 1) as edge
        from per_zone z join top using (player_id) join lg_zone l using (zone)
        left join (select stats_player_id, full_name from dim_player where league = '{lg}' and is_current) p
            on p.stats_player_id = z.player_id
    ''').df()
    overall = (players.assign(w=players['attempts'] * players['edge'])
               .groupby('player').agg(attempts=('attempts', 'sum'), weighted=('w', 'sum')))
    overall['edge_pp100'] = (overall['weighted'] / overall['attempts']).round(1)
    table = (players.pivot_table(index='player', columns='zone', values='edge', aggfunc='first')
             .join(overall[['attempts', 'edge_pp100']]).sort_values('edge_pp100', ascending=False))
    edges[lg] = table
    print(LABEL[lg]); display(table.round(1))

In [ ]:
for lg in LEAGUES:
    season, note, z = zones[lg]
    best, worst = z.iloc[0], z.iloc[-1]
    threes = z[z['zone'].str.contains('3')]
    top = edges[lg].reset_index().head(5)
    findings[lg] = {
        'headline': f"In {season}, a {best['zone']} attempt is worth {best['pp100']:.0f} points per 100 shots, against {worst['pp100']:.0f} from {worst['zone']}.",
        'facts': [
            {'label': 'Season', 'value': season + (f' ({note})' if note else '')},
            {'label': 'Best zone', 'value': f"{best['zone']}: {best['pp100']:.1f} pts / 100"},
            {'label': 'Share of shots from three', 'value': f"{threes['share_pct'].sum():.1f}%"},
            {'label': 'Biggest edge among volume shooters', 'value': f"{top.iloc[0]['player']}: {top.iloc[0]['edge_pp100']:+.1f} pts / 100" if len(top) else 'n/a'},
        ],
        'table': {
            'columns': ['Player', 'Attempts', 'Edge (pts / 100)'],
            'rows': [[r['player'], int(r['attempts']), float(r['edge_pp100'])] for _, r in top.iterrows()],
        },
    }
save_findings('01_shot_profiles', {'title': 'Shot profiles', 'leagues': findings})
findings